In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, learning_curve, train_test_split
from sklearn.metrics import classification_report

sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')

# detectar se ta rodando no colab ou local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    base_dir = '/content/tcc-analise-sentimento'
else:
    # Local (VS Code) - ajusta diretorio raiz
    if 'src' in os.getcwd():
        os.chdir('..')
    base_dir = os.getcwd()

caminho_csv = os.path.join(base_dir, 'dados', 'processado', 'synthetic_dataset_apps.csv')
dir_resultados = os.path.join(base_dir, 'resultados_apps')
os.makedirs(dir_resultados, exist_ok=True)

if os.path.exists(caminho_csv):
    df = pd.read_csv(caminho_csv)
    df = df.dropna(subset=['frase_limpa'])
    X = df['frase_limpa']
    y = df['classe']
    print(f'Dataset APPS carregado: {len(df)} amostras')
else:
    print(f"⚠️ AVISO: Arquivo {caminho_csv} não encontrado.")
    print("Execute o notebook 'src/01_apps_training.ipynb' primeiro para gerar o dataset processado.")

### VALIDAÇÃO CRUZADA E ROBUSTEZ (APPS)
Calcula as métricas médias para garantir que os resultados não dependam de um split específico.

In [ ]:
if 'X' in locals():
    def criar_pipeline(clf):
        return Pipeline([('tfidf', TfidfVectorizer(max_features=5000)), ('clf', clf)])

    modelos = [
        ('Naive Bayes', MultinomialNB()),
        ('Reg. Logística', LogisticRegression(max_iter=1000))
    ]

    for nome, clf in modelos:
        scores = cross_val_score(criar_pipeline(clf), X, y, cv=5, scoring='f1_weighted')
        print(f"{nome}: F1 Médio = {scores.mean():.2%} (+/- {scores.std():.2%})")
else:
    print("⚠️ Carregue o dataset na primeira célula para rodar a análise.")

## 💾 Backup automático (Colab → download zip)

Se rodando no Colab, comprime `resultados/` em zip e baixa pro PC.
Se rodando local, não faz nada (arquivos já estão no disco).

**Sempre rode esta célula ANTES de fechar o Colab** — senão você perde tudo.

In [ ]:
# ============================================================
# CÉLULA FINAL — Backup automático Colab → download
# Local: no-op | Colab: zip + download de resultados/
# ============================================================
import os, sys
if 'IN_COLAB' not in globals():
    IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import shutil, time
    from google.colab import files

    if 'RESULTADOS' not in globals():
        RESULTADOS = '/content/tcc-analise-sentimento/resultados'

    timestamp = time.strftime('%Y%m%d_%H%M%S')
    nome_zip = f'resultados_colab_{timestamp}'
    zip_path = shutil.make_archive(f'/content/{nome_zip}', 'zip', RESULTADOS)
    print(f'Comprimido: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')
    print('Iniciando download... (descompacte em resultados/ no PC local)')
    files.download(zip_path)
else:
    pasta = RESULTADOS if 'RESULTADOS' in globals() else 'resultados/'
    print(f'Ambiente local - resultados ja salvos em: {pasta}')
    print('Nenhuma acao necessaria.')
